In [1]:
import os
import time
import numpy as np
import pandas as pd
import multiprocessing
import gc

# ML e Métricas
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, f1_score, precision_score, 
                             roc_auc_score, matthews_corrcoef, precision_recall_curve, 
                             auc, average_precision_score)

# Deep Learning para o Surrogate Model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method

# ==========================================
# PAINEL DE CONTROLE DO EXPERIMENTO (DECISION TREE)
# ==========================================
DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15'] # Adicione 'UNSW-NB15' para rodar ambos
RUN_BINARY = True          
RUN_MULTICLASS = True      

# CONTROLE DE ATAQUES
ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2'] 
EPSILON_LINF = 0.3   
EPSILON_L2 = 3.0     

# Criação das pastas de exportação
os.makedirs('relatorios final', exist_ok=True)
os.makedirs('curves_data', exist_ok=True) 

print(f"Datasets Selecionados: {DATASETS_TO_RUN}")
print(f"Modos Ativados: Binário={RUN_BINARY} | Multiclasse={RUN_MULTICLASS}")
print(f"Ataques Ativados: {ATTACKS_TO_RUN}")

Datasets Selecionados: ['Bot-IoT', 'UNSW-NB15']
Modos Ativados: Binário=True | Multiclasse=True
Ataques Ativados: ['FGSM', 'RANDOM_LINF', 'RANDOM_L2']


In [2]:
# ==========================================
# FUNÇÕES DE MODELAÇÃO E MÉTRICAS
# ==========================================
def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model

def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}
    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0
        
    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None
    
    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)
    metrics[f'{context_name}_Precision'] = precision_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_Recall'] = recall_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_F1'] = f1_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_MCC'] = matthews_corrcoef(y_true, y_pred)
    
    mask_normal = (y_true == normal_idx)
    mask_attack = (y_true != normal_idx)
    
    if np.sum(mask_normal) > 0:
        metrics[f'{context_name}_FAR'] = np.sum((y_pred != normal_idx) & mask_normal) / np.sum(mask_normal)
    else:
        metrics[f'{context_name}_FAR'] = 0.0

    if np.sum(mask_attack) > 0:
        metrics[f'{context_name}_ASR'] = np.sum((y_pred == normal_idx) & mask_attack) / np.sum(mask_attack)
    else:
        metrics[f'{context_name}_ASR'] = 0.0

    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)
            prec, rec, _ = precision_recall_curve(y_true, prob_positive)
            metrics[f'{context_name}_PR_AUC'] = auc(rec, prec)
        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
            y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
            metrics[f'{context_name}_PR_AUC'] = average_precision_score(y_true_bin, y_prob, average="macro")
    except Exception as e:
        metrics[f'{context_name}_AUC'] = 0.0
        metrics[f'{context_name}_PR_AUC'] = 0.0
    
    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    for idx, name in enumerate(class_names):
        if idx == normal_idx: continue
        mask_t = (y_true == idx)
        if np.sum(mask_t) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum((y_pred == normal_idx) & mask_t) / np.sum(mask_t)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0
            
    return metrics

In [3]:
# ==========================================
# LOOP PRINCIPAL DO EXPERIMENTO (DECISION TREE)
# ==========================================
for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*50}")
    print(f">>> A INICIAR EXPERIMENTOS: {dataset_name.upper()}")
    print(f"{'='*50}")
    
    # --- Limpeza de CSVs Antigos do Mesmo Dataset para evitar duplicações no Append ---
    for f in os.listdir('relatorios final'):
        if f.startswith(f'decision_tree_{dataset_name}'):
            os.remove(os.path.join('relatorios final', f))
    
    # 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
    if dataset_name == 'Bot-IoT':
        df_train = pd.read_csv("data2/BotIoT_training-set.csv")
        df_test = pd.read_csv("data2/BotIoT_testing-set.csv")
    elif dataset_name == 'UNSW-NB15':
        df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("data/UNSW_NB15_testing-set.csv")
        
    for df in [df_train, df_test]:
        if 'id' in df.columns: df.drop(columns=['id'], inplace=True)

    y_train_bin = df_train['label'].values
    y_test_bin = df_test['label'].values
    class_names_bin = ['Normal', 'Attack']

    df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
    df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))
    y_train_multi = le.transform(df_train['attack_cat'])
    y_test_multi = le.transform(df_test['attack_cat'])
    class_names_multi = le.classes_

    df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
    df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

    categorical_cols = df_train.select_dtypes(include=['object']).columns
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

    preprocessor = ColumnTransformer([
        ('num', MinMaxScaler(feature_range=(0,1)), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

    print(">>> A aplicar Scaling e One-Hot Encoding...")
    preprocessor.fit(df_train)
    X_train = preprocessor.transform(df_train).astype('float32')
    X_test = preprocessor.transform(df_test).astype('float32')

    del df_train, df_test
    gc.collect()

    # 2. GERAÇÃO DOS ATAQUES ADVERSARIAIS E RUÍDOS
    attacks_dict_bin = {}
    attacks_dict_multi = {}

    if 'FGSM' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque FGSM (Caixa-Branca Transferida)...")
        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['FGSM'] = fast_gradient_method(logits_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bin, logits_bin
        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['FGSM'] = fast_gradient_method(logits_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_multi, logits_multi
        gc.collect()

    if 'RANDOM_LINF' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_infinito (Eps={EPSILON_LINF})...")
        noise = np.random.uniform(-EPSILON_LINF, EPSILON_LINF, X_test.shape).astype('float32')
        if RUN_BINARY: attacks_dict_bin['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    if 'RANDOM_L2' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_2 (Eps={EPSILON_L2})...")
        noise = np.random.normal(0, 1, X_test.shape).astype('float32')
        norms = np.linalg.norm(noise, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        noise = noise * (EPSILON_L2 / norms)
        if RUN_BINARY: attacks_dict_bin['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise
    
    gc.collect()

    # 3. HIPERPARÂMETROS DA DECISION TREE
    dt_param_grid = {
        'criterion': ['gini'],
        'max_depth': [10],
        'min_samples_leaf': [1],
        'min_samples_split': [2]
    }

    modes_to_run = []
    if RUN_BINARY: modes_to_run.append('binary')
    if RUN_MULTICLASS: modes_to_run.append('multiclass')

    # 4. EXECUÇÃO DOS MODELOS (Com Salvamento Incremental)
    for mode in modes_to_run:
        print(f"\n  -> 🚀 A correr Pipeline {mode.upper()}...")
        
        csv_name = f'relatorios final/decision_tree_{dataset_name}_{mode}.csv'

        if mode == 'binary':
            y_train_curr, y_test_curr = y_train_bin, y_test_bin
            active_adv_dict = attacks_dict_bin
            class_names_curr = class_names_bin
        else:
            y_train_curr, y_test_curr = y_train_multi, y_test_multi
            active_adv_dict = attacks_dict_multi
            class_names_curr = class_names_multi

        for crit in dt_param_grid['criterion']:
            for depth in dt_param_grid['max_depth']:
                for min_leaf in dt_param_grid['min_samples_leaf']:
                    for min_split in dt_param_grid['min_samples_split']:
                        
                        print(f"     [Crit={crit} | Depth={depth} | MinLeaf={min_leaf}] A treinar DT...")
                        model = DecisionTreeClassifier(
                            criterion=crit, 
                            max_depth=depth, 
                            min_samples_leaf=min_leaf,
                            min_samples_split=min_split, 
                            random_state=1 
                        )
                        
                        t0 = time.time()
                        model.fit(X_train, y_train_curr)
                        train_time = time.time() - t0
                        
                        # Inferência Limpa
                        t1 = time.time()
                        yp_clean = model.predict(X_test)
                        yp_clean_prob = model.predict_proba(X_test)
                        infer_time_clean = time.time() - t1
                        m_clean = calculate_miss_rates(y_test_curr, yp_clean, yp_clean_prob, "Clean", class_names_curr)
                        
                        # Iterando sobre os ataques
                        for atk_name, X_adv_curr in active_adv_dict.items():
                            t2 = time.time()
                            yp_adv = model.predict(X_adv_curr)
                            yp_adv_prob = model.predict_proba(X_adv_curr)
                            infer_time_adv = time.time() - t2
                            
                            m_adv = calculate_miss_rates(y_test_curr, yp_adv, yp_adv_prob, "Adv", class_names_curr)
                            acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']
                            
                            row = {
                                'Dataset': dataset_name,
                                'Attack': atk_name,
                                'Criterion': crit, 
                                'MaxDepth': depth, 
                                'MinLeaf': min_leaf,       
                                'MinSplit': min_split,     
                                'TrainTime_s': train_time, 
                                'InferTime_Adv_s': infer_time_adv,    
                                'Acc_Drop_pp': acc_drop*100
                            }
                            row.update(m_clean)
                            row.update(m_adv)
                            
                            # ---> SALVAMENTO INCREMENTAL NO CSV <---
                            row_df = pd.DataFrame([row])
                            file_exists = os.path.exists(csv_name)
                            row_df.to_csv(csv_name, mode='a', header=not file_exists, index=False)
                            
                            # ---> SALVAMENTO DO .NPZ <---
                            file_tag = f"dt_{dataset_name}_{mode}_{atk_name}_D{depth}_L{min_leaf}_S{min_split}"
                            np.savez(f"curves_data/{file_tag}.npz", 
                                     model_name="Decision Tree",
                                     attack_name=atk_name,
                                     y_true=y_test_curr, 
                                     y_prob_clean=yp_clean_prob, 
                                     y_prob_adv=yp_adv_prob,
                                     class_names=class_names_curr)
                            
                            del yp_adv, yp_adv_prob, m_adv, row, row_df
                        
                        del model, yp_clean, yp_clean_prob, m_clean
                        gc.collect()

    del attacks_dict_bin, attacks_dict_multi, X_train, X_test
    gc.collect()

print("\n🚀 EXPERIMENTO DT CONCLUÍDO COM SUCESSO!")


>>> A INICIAR EXPERIMENTOS: BOT-IOT


C:\Users\root.REDE-LAGESED\AppData\Local\Temp\ipykernel_33320\2524664148.py:40: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include=['object']).columns


>>> A aplicar Scaling e One-Hot Encoding...
>>> A gerar Ataque FGSM (Caixa-Branca Transferida)...
>>> A gerar Ruído Aleatório L_infinito (Eps=0.3)...
>>> A gerar Ruído Aleatório L_2 (Eps=3.0)...

  -> 🚀 A correr Pipeline BINARY...
     [Crit=gini | Depth=10 | MinLeaf=1] A treinar DT...

  -> 🚀 A correr Pipeline MULTICLASS...
     [Crit=gini | Depth=10 | MinLeaf=1] A treinar DT...

>>> A INICIAR EXPERIMENTOS: UNSW-NB15


C:\Users\root.REDE-LAGESED\AppData\Local\Temp\ipykernel_33320\2524664148.py:40: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include=['object']).columns


>>> A aplicar Scaling e One-Hot Encoding...
>>> A gerar Ataque FGSM (Caixa-Branca Transferida)...
>>> A gerar Ruído Aleatório L_infinito (Eps=0.3)...
>>> A gerar Ruído Aleatório L_2 (Eps=3.0)...

  -> 🚀 A correr Pipeline BINARY...
     [Crit=gini | Depth=10 | MinLeaf=1] A treinar DT...

  -> 🚀 A correr Pipeline MULTICLASS...
     [Crit=gini | Depth=10 | MinLeaf=1] A treinar DT...

🚀 EXPERIMENTO DT CONCLUÍDO COM SUCESSO!
